<a href="https://colab.research.google.com/github/tanercc/python-colab/blob/dev/tjk_to_Predict_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
!pip install -q mysql-connector-python
!pip install pandas tensorflow

Get TJK DB

In [24]:
import mysql.connector as connection
import pandas as pd
try:
    mydb = connection.connect(host="taner.web.tr", database = 'tanerweb_tjk',user="tanerweb_tjk", passwd="Ka9jVjRJRtRW",use_pure=True)
    query = """
SELECT
`atlar`.`KOD` AS `code`,
`atlar`.`ADKUCUK` AS `name`,
`kosular`.`GRUP_EN` AS `group`,
`atlar`.`ANNE` AS `mother`,
`atlar`.`BABA` AS `father`,
`kosular`.`TARIH` AS `date`,
`hipodrom`.`AD` AS `hipname`,
`hipodrom`.`KOD` AS `hipcode`,
`kosular`.`ActiveClass` AS `surface`,
COALESCE(CASE WHEN `kosular`.`PIST` = 'cim' THEN `hava`.`CIM_EN` ELSE `hava`.`KUM_EN` END, 'Normal') AS `ground`,
`kosular`.`RACENO` AS `raceno`,
`atlar`.`KILO` + `atlar`.`FAZLAKILO` AS `weight`,
`atlar`.`JOKEYADI` AS `jockey`,
`kosular`.`MESAFE` AS `metre`,
`atlar`.`DERECE` AS `time`,
COALESCE(`atlar`.`SONUC`, (SELECT COUNT(NO) FROM `atlar` AS `atsay` WHERE `atlar`.`TARIHKOD`=`atsay`.`TARIHKOD` AND `atlar`.`HIPODROMKOD`=`atsay`.`HIPODROMKOD` AND `atlar`.`KOSUNO`=`atsay`.`KOSUNO`)) AS `pos`
FROM `atlar`
LEFT JOIN `hava` ON (`atlar`.`TARIHKOD` = `hava`.`TARIHKOD` AND `atlar`.`HIPODROMKOD` = `hava`.`HIPODROMKOD`)
LEFT JOIN `kosular` ON (`kosular`.`TARIHKOD` = `atlar`.`TARIHKOD` AND `atlar`.`HIPODROMKOD` = `kosular`.`HIPODROMKOD` AND `atlar`.`KOSUNO` = `kosular`.`NO`)
LEFT JOIN `hipodrom` ON `hipodrom`.`KOD` = `atlar`.`HIPODROMKOD`
WHERE `atlar`.`TARIHKOD` > 0
AND `atlar`.`KILO` > 0
AND `atlar`.`KOSMAZ` = 0
AND `atlar`.`GECCIKIS_BOY` = ''
AND `atlar`.`START` > 0
AND `atlar`.`DERECE` > 0
AND `atlar`.`SONUC` > 0
ORDER BY `atlar`.`TARIHKOD` DESC,`atlar`.`HIPODROMKOD`,`atlar`.`KOSUNO`
    """
    #AND `atlar`.`HIPODROMKOD` = 1
    df = pd.read_sql(query,mydb)
    mydb.close() #close the connection
except Exception as e:
    mydb.close()
    print(str(e))

<ipython-input-24-94ac3bfee369>:37: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query,mydb)


In [25]:
df.head(10)

,code,name,group,mother,father,date,hipname,hipcode,surface,ground,raceno,weight,jockey,metre,time,pos
0,105613,Super Plane,3 Years Old Thoroughbreds,SILVER PLANE (USA),SUPER SAVER (USA),10/04/2025,İzmir Şirinyer Hipodromu,2,sand,Good Going,1,62.0,HALEF KATI,1900,127.52,3
1,105542,Crazygold,3 Years Old Thoroughbreds,SERPE,VICTORY GALLOP (CAN),10/04/2025,İzmir Şirinyer Hipodromu,2,sand,Good Going,1,60.0,BEKİR MERT MIRIK,1900,126.83,2
2,102818,Always A Reason,3 Years Old Thoroughbreds,REASON WHY,BLUEGRASS CAT (USA),10/04/2025,İzmir Şirinyer Hipodromu,2,sand,Good Going,1,58.0,ONUR YILDIZ,1900,128.98,5
3,105021,Aydos Kız,3 Years Old Thoroughbreds,VICTORIUM,TOCCET (USA),10/04/2025,İzmir Şirinyer Hipodromu,2,sand,Good Going,1,58.0,ERHAN AKTUĞ,1900,127.83,4
4,104562,Başpınar,3 Years Old Thoroughbreds,TONNANTE (GB),GAZIBORA KHAN (GB),10/04/2025,İzmir Şirinyer Hipodromu,2,sand,Good Going,1,57.0,ALPEREN OLUK,1900,129.89,6
5,105604,Midnight Bourbon,3 Years Old Thoroughbreds,SALLY OBELISK,GRAYSTORM,10/04/2025,İzmir Şirinyer Hipodromu,2,sand,Good Going,1,58.0,MUSTAFA ÇİÇEK,1900,126.39,1
6,102810,Prenses Cemre,3 Years Old Thoroughbreds,AY IŞIĞI,TOCCET (USA),10/04/2025,İzmir Şirinyer Hipodromu,2,sand,Good Going,1,58.0,TUGAY ALICI,1900,129.98,7
7,103412,Batımert,3 Years Old Thoroughbreds,YALÇINKIZ,EUPRHATES (USA),10/04/2025,İzmir Şirinyer Hipodromu,2,sand,Good Going,2,58.0,MUHAMMED MİR BİLGİN,1400,89.16,2
8,103069,Son Of Kınowa,3 Years Old Thoroughbreds,PRENSES LEIA,KINOWA,10/04/2025,İzmir Şirinyer Hipodromu,2,sand,Good Going,2,58.0,MUSTAFA ÇİÇEK,1400,89.86,5
9,111089,Kurtuluş Ağa,3 Years Old Thoroughbreds,FATO ANA,MARCAVELLY (USA),10/04/2025,İzmir Şirinyer Hipodromu,2,sand,Good Going,2,56.3,BEKİR MERT MIRIK,1400,92.17,7


In [26]:
#df.to_csv("data-output.csv")

In [27]:
import pandas as pd

# STEP 1: Calculate breed scores for father
father_stats = df.groupby("father").agg(
    race_count=("pos", "count"),
    avg_pos=("pos", "mean"),
    avg_time=("time", "mean"),
    win_count=("pos", lambda x: (x == 1).sum())
)
father_stats["win_rate"] = father_stats["win_count"] / father_stats["race_count"]
father_stats["breed_score"] = father_stats["avg_pos"] * 0.5 + father_stats["avg_time"] * 0.5

# STEP 2: Calculate breed scores for mother
mother_stats = df.groupby("mother").agg(
    race_count=("pos", "count"),
    avg_pos=("pos", "mean"),
    avg_time=("time", "mean"),
    win_count=("pos", lambda x: (x == 1).sum())
)
mother_stats["win_rate"] = mother_stats["win_count"] / mother_stats["race_count"]
mother_stats["breed_score"] = mother_stats["avg_pos"] * 0.5 + mother_stats["avg_time"] * 0.5

# STEP 3: Map scores back to original dataframe
df["father_breed_score"] = df["father"].map(father_stats["breed_score"])
df["father_win_rate"] = df["father"].map(father_stats["win_rate"])
df["mother_breed_score"] = df["mother"].map(mother_stats["breed_score"])
df["mother_win_rate"] = df["mother"].map(mother_stats["win_rate"])

# DONE! Preview
print(df[[
    "name", "mother", "father",
    "father_breed_score", "father_win_rate", "mother_breed_score", "mother_win_rate"
]])

# Show result
df.head()

                   name              mother                father  \
0           Super Plane  SILVER PLANE (USA)     SUPER SAVER (USA)   
1             Crazygold               SERPE  VICTORY GALLOP (CAN)   
2       Always A Reason          REASON WHY   BLUEGRASS CAT (USA)   
3             Aydos Kız           VICTORIUM          TOCCET (USA)   
4              Başpınar       TONNANTE (GB)    GAZIBORA KHAN (GB)   
...                 ...                 ...                   ...   
293732           My Cem             Prelude         Zanjero (Usa)   
293733     Black Legacy           Holy Rock          Mendıp (Usa)   
293734       Fatih Reis       Çeli̇k Sultan       Talıp Han (Ire)   
293735         Menachem       Sılver Dancer     Dream Ahead (Usa)   
293736     Oğlum Hikmet  Polar Cadeaux (Gb)  Vıctory Gallop (Can)   

        father_breed_score  father_win_rate  mother_breed_score  \
0                53.333411         0.130653           54.101667   
1                57.964352         0.

,code,name,group,mother,father,date,hipname,hipcode,surface,ground,raceno,weight,jockey,metre,time,pos,father_breed_score,father_win_rate,mother_breed_score,mother_win_rate
0,105613,Super Plane,3 Years Old Thoroughbreds,SILVER PLANE (USA),SUPER SAVER (USA),10/04/2025,İzmir Şirinyer Hipodromu,2,sand,Good Going,1,62.0,HALEF KATI,1900,127.52,3,53.333411,0.130653,54.101667,0.080000
1,105542,Crazygold,3 Years Old Thoroughbreds,SERPE,VICTORY GALLOP (CAN),10/04/2025,İzmir Şirinyer Hipodromu,2,sand,Good Going,1,60.0,BEKİR MERT MIRIK,1900,126.83,2,57.964352,0.124941,53.086346,0.115385
2,102818,Always A Reason,3 Years Old Thoroughbreds,REASON WHY,BLUEGRASS CAT (USA),10/04/2025,İzmir Şirinyer Hipodromu,2,sand,Good Going,1,58.0,ONUR YILDIZ,1900,128.98,5,51.502059,0.159322,54.447500,0.000000
3,105021,Aydos Kız,3 Years Old Thoroughbreds,VICTORIUM,TOCCET (USA),10/04/2025,İzmir Şirinyer Hipodromu,2,sand,Good Going,1,58.0,ERHAN AKTUĞ,1900,127.83,4,57.103540,0.112340,55.250000,0.000000
4,104562,Başpınar,3 Years Old Thoroughbreds,TONNANTE (GB),GAZIBORA KHAN (GB),10/04/2025,İzmir Şirinyer Hipodromu,2,sand,Good Going,1,57.0,ALPEREN OLUK,1900,129.89,6,49.296867,0.120000,61.299512,0.073171


In [28]:
import pandas as pd

# Helper: create a breed_score column per row
df["breed_score_row"] = df["pos"] * 0.5 + df["time"] * 0.5

# STEP 1 — MOTHER-based sibling stats
mother_stats = df.groupby("mother").agg(
    sibling_mother_score=("breed_score_row", "mean"),
    sibling_mother_win_rate=("pos", lambda x: (x == 1).sum() / len(x))
)

# STEP 2 — FATHER-based sibling stats
father_stats = df.groupby("father").agg(
    sibling_father_score=("breed_score_row", "mean"),
    sibling_father_win_rate=("pos", lambda x: (x == 1).sum() / len(x))
)

# STEP 3 — Map them to the main DataFrame
df["sibling_mother_score"] = df["mother"].map(mother_stats["sibling_mother_score"])
df["sibling_mother_win_rate"] = df["mother"].map(mother_stats["sibling_mother_win_rate"])
df["sibling_father_score"] = df["father"].map(father_stats["sibling_father_score"])
df["sibling_father_win_rate"] = df["father"].map(father_stats["sibling_father_win_rate"])

# STEP 4 — Combine scores (optional, average of both sides)
df["sibling_breed_score"] = (
    df["sibling_mother_score"] + df["sibling_father_score"]
) / 2

df["sibling_win_rate"] = (
    df["sibling_mother_win_rate"] + df["sibling_father_win_rate"]
) / 2

# DONE! Preview
print(df[[
    "name", "mother", "father",
    "sibling_breed_score", "sibling_win_rate"
]])

# Show result
df.head()

                   name              mother                father  \
0           Super Plane  SILVER PLANE (USA)     SUPER SAVER (USA)   
1             Crazygold               SERPE  VICTORY GALLOP (CAN)   
2       Always A Reason          REASON WHY   BLUEGRASS CAT (USA)   
3             Aydos Kız           VICTORIUM          TOCCET (USA)   
4              Başpınar       TONNANTE (GB)    GAZIBORA KHAN (GB)   
...                 ...                 ...                   ...   
293732           My Cem             Prelude         Zanjero (Usa)   
293733     Black Legacy           Holy Rock          Mendıp (Usa)   
293734       Fatih Reis       Çeli̇k Sultan       Talıp Han (Ire)   
293735         Menachem       Sılver Dancer     Dream Ahead (Usa)   
293736     Oğlum Hikmet  Polar Cadeaux (Gb)  Vıctory Gallop (Can)   

        sibling_breed_score  sibling_win_rate  
0                 53.717539          0.105327  
1                 55.525349          0.120163  
2                 52.974780

,code,name,group,mother,father,date,hipname,hipcode,surface,ground,...,father_win_rate,mother_breed_score,mother_win_rate,breed_score_row,sibling_mother_score,sibling_mother_win_rate,sibling_father_score,sibling_father_win_rate,sibling_breed_score,sibling_win_rate
0,105613,Super Plane,3 Years Old Thoroughbreds,SILVER PLANE (USA),SUPER SAVER (USA),10/04/2025,İzmir Şirinyer Hipodromu,2,sand,Good Going,...,0.130653,54.101667,0.080000,65.260,54.101667,0.080000,53.333411,0.130653,53.717539,0.105327
1,105542,Crazygold,3 Years Old Thoroughbreds,SERPE,VICTORY GALLOP (CAN),10/04/2025,İzmir Şirinyer Hipodromu,2,sand,Good Going,...,0.124941,53.086346,0.115385,64.415,53.086346,0.115385,57.964352,0.124941,55.525349,0.120163
2,102818,Always A Reason,3 Years Old Thoroughbreds,REASON WHY,BLUEGRASS CAT (USA),10/04/2025,İzmir Şirinyer Hipodromu,2,sand,Good Going,...,0.159322,54.447500,0.000000,66.990,54.447500,0.000000,51.502059,0.159322,52.974780,0.079661
3,105021,Aydos Kız,3 Years Old Thoroughbreds,VICTORIUM,TOCCET (USA),10/04/2025,İzmir Şirinyer Hipodromu,2,sand,Good Going,...,0.112340,55.250000,0.000000,65.915,55.250000,0.000000,57.103540,0.112340,56.176770,0.056170
4,104562,Başpınar,3 Years Old Thoroughbreds,TONNANTE (GB),GAZIBORA KHAN (GB),10/04/2025,İzmir Şirinyer Hipodromu,2,sand,Good Going,...,0.120000,61.299512,0.073171,67.945,61.299512,0.073171,49.296867,0.120000,55.298189,0.096585


In [29]:
# Combine all breed scores into one
df["combined_breed_score"] = df[[
    "father_breed_score",
    "mother_breed_score",
    "sibling_breed_score"
]].mean(axis=1)

# Optional: round for readability
df["combined_breed_score"] = df["combined_breed_score"].round(2)

# Preview
print(df[[
    "name", "father_breed_score", "mother_breed_score", "sibling_breed_score", "combined_breed_score"
]])

# Show result
df.head()

                   name  father_breed_score  mother_breed_score  \
0           Super Plane           53.333411           54.101667   
1             Crazygold           57.964352           53.086346   
2       Always A Reason           51.502059           54.447500   
3             Aydos Kız           57.103540           55.250000   
4              Başpınar           49.296867           61.299512   
...                 ...                 ...                 ...   
293732           My Cem           54.788408           60.785000   
293733     Black Legacy           51.442472           58.555000   
293734       Fatih Reis           51.705430           59.339583   
293735         Menachem           61.269000           52.060667   
293736     Oğlum Hikmet           55.505467           68.781875   

        sibling_breed_score  combined_breed_score  
0                 53.717539                 53.72  
1                 55.525349                 55.53  
2                 52.974780            

,code,name,group,mother,father,date,hipname,hipcode,surface,ground,...,mother_breed_score,mother_win_rate,breed_score_row,sibling_mother_score,sibling_mother_win_rate,sibling_father_score,sibling_father_win_rate,sibling_breed_score,sibling_win_rate,combined_breed_score
0,105613,Super Plane,3 Years Old Thoroughbreds,SILVER PLANE (USA),SUPER SAVER (USA),10/04/2025,İzmir Şirinyer Hipodromu,2,sand,Good Going,...,54.101667,0.080000,65.260,54.101667,0.080000,53.333411,0.130653,53.717539,0.105327,53.72
1,105542,Crazygold,3 Years Old Thoroughbreds,SERPE,VICTORY GALLOP (CAN),10/04/2025,İzmir Şirinyer Hipodromu,2,sand,Good Going,...,53.086346,0.115385,64.415,53.086346,0.115385,57.964352,0.124941,55.525349,0.120163,55.53
2,102818,Always A Reason,3 Years Old Thoroughbreds,REASON WHY,BLUEGRASS CAT (USA),10/04/2025,İzmir Şirinyer Hipodromu,2,sand,Good Going,...,54.447500,0.000000,66.990,54.447500,0.000000,51.502059,0.159322,52.974780,0.079661,52.97
3,105021,Aydos Kız,3 Years Old Thoroughbreds,VICTORIUM,TOCCET (USA),10/04/2025,İzmir Şirinyer Hipodromu,2,sand,Good Going,...,55.250000,0.000000,65.915,55.250000,0.000000,57.103540,0.112340,56.176770,0.056170,56.18
4,104562,Başpınar,3 Years Old Thoroughbreds,TONNANTE (GB),GAZIBORA KHAN (GB),10/04/2025,İzmir Şirinyer Hipodromu,2,sand,Good Going,...,61.299512,0.073171,67.945,61.299512,0.073171,49.296867,0.120000,55.298189,0.096585,55.30


In [30]:
# STEP 1 — Calculate jockey performance stats
jockey_stats = df.groupby("jockey").agg(
    jockey_avg_pos=("pos", "mean"),
    jockey_avg_time=("time", "mean"),
    jockey_win_count=("pos", lambda x: (x == 1).sum()),
    race_count=("pos", "count")
)

# STEP 2 — Add win rate and jockey score
jockey_stats["jockey_win_rate"] = jockey_stats["jockey_win_count"] / jockey_stats["race_count"]
jockey_stats["jockey_score"] = (
    jockey_stats["jockey_avg_pos"] * 0.5 + jockey_stats["jockey_avg_time"] * 0.5
)

# STEP 3 — Map stats back to DataFrame
df["jockey_score"] = df["jockey"].map(jockey_stats["jockey_score"])
df["jockey_win_rate"] = df["jockey"].map(jockey_stats["jockey_win_rate"])

# Optional: round for readability
df["jockey_score"] = df["jockey_score"].round(2)
df["jockey_win_rate"] = df["jockey_win_rate"].round(2)

# Preview
print(df[["name", "jockey", "jockey_score", "jockey_win_rate"]])

# Show result
df.head()

                   name              jockey  jockey_score  jockey_win_rate
0           Super Plane          HALEF KATI         55.15             0.06
1             Crazygold    BEKİR MERT MIRIK         54.78             0.12
2       Always A Reason         ONUR YILDIZ         56.51             0.12
3             Aydos Kız         ERHAN AKTUĞ         55.11             0.11
4              Başpınar        ALPEREN OLUK         56.61             0.07
...                 ...                 ...           ...              ...
293732           My Cem         Abdullah Ar         65.54             0.00
293733     Black Legacy       İlhami̇ Göçer         54.18             0.00
293734       Fatih Reis  Mehmet Emi̇n Doğan         58.09             0.11
293735         Menachem         İsa Akyavuz         56.45             0.13
293736     Oğlum Hikmet         Mahmut Ünlü         55.05             0.06

[293737 rows x 4 columns]


,code,name,group,mother,father,date,hipname,hipcode,surface,ground,...,breed_score_row,sibling_mother_score,sibling_mother_win_rate,sibling_father_score,sibling_father_win_rate,sibling_breed_score,sibling_win_rate,combined_breed_score,jockey_score,jockey_win_rate
0,105613,Super Plane,3 Years Old Thoroughbreds,SILVER PLANE (USA),SUPER SAVER (USA),10/04/2025,İzmir Şirinyer Hipodromu,2,sand,Good Going,...,65.260,54.101667,0.080000,53.333411,0.130653,53.717539,0.105327,53.72,55.15,0.06
1,105542,Crazygold,3 Years Old Thoroughbreds,SERPE,VICTORY GALLOP (CAN),10/04/2025,İzmir Şirinyer Hipodromu,2,sand,Good Going,...,64.415,53.086346,0.115385,57.964352,0.124941,55.525349,0.120163,55.53,54.78,0.12
2,102818,Always A Reason,3 Years Old Thoroughbreds,REASON WHY,BLUEGRASS CAT (USA),10/04/2025,İzmir Şirinyer Hipodromu,2,sand,Good Going,...,66.990,54.447500,0.000000,51.502059,0.159322,52.974780,0.079661,52.97,56.51,0.12
3,105021,Aydos Kız,3 Years Old Thoroughbreds,VICTORIUM,TOCCET (USA),10/04/2025,İzmir Şirinyer Hipodromu,2,sand,Good Going,...,65.915,55.250000,0.000000,57.103540,0.112340,56.176770,0.056170,56.18,55.11,0.11
4,104562,Başpınar,3 Years Old Thoroughbreds,TONNANTE (GB),GAZIBORA KHAN (GB),10/04/2025,İzmir Şirinyer Hipodromu,2,sand,Good Going,...,67.945,61.299512,0.073171,49.296867,0.120000,55.298189,0.096585,55.30,56.61,0.07


In [31]:
# Drop unnecessary columns (names, codes, date, etc.)
df = df.drop(columns=['father_breed_score', 'father_win_rate', 'mother_breed_score', 'mother_win_rate', 'breed_score_row', 'sibling_mother_score', 'sibling_mother_win_rate', 'sibling_father_score', 'sibling_father_win_rate', 'sibling_breed_score', 'sibling_win_rate', 'jockey_win_rate'])

# Preview
df.head()


,code,name,group,mother,father,date,hipname,hipcode,surface,ground,raceno,weight,jockey,metre,time,pos,combined_breed_score,jockey_score
0,105613,Super Plane,3 Years Old Thoroughbreds,SILVER PLANE (USA),SUPER SAVER (USA),10/04/2025,İzmir Şirinyer Hipodromu,2,sand,Good Going,1,62.0,HALEF KATI,1900,127.52,3,53.72,55.15
1,105542,Crazygold,3 Years Old Thoroughbreds,SERPE,VICTORY GALLOP (CAN),10/04/2025,İzmir Şirinyer Hipodromu,2,sand,Good Going,1,60.0,BEKİR MERT MIRIK,1900,126.83,2,55.53,54.78
2,102818,Always A Reason,3 Years Old Thoroughbreds,REASON WHY,BLUEGRASS CAT (USA),10/04/2025,İzmir Şirinyer Hipodromu,2,sand,Good Going,1,58.0,ONUR YILDIZ,1900,128.98,5,52.97,56.51
3,105021,Aydos Kız,3 Years Old Thoroughbreds,VICTORIUM,TOCCET (USA),10/04/2025,İzmir Şirinyer Hipodromu,2,sand,Good Going,1,58.0,ERHAN AKTUĞ,1900,127.83,4,56.18,55.11
4,104562,Başpınar,3 Years Old Thoroughbreds,TONNANTE (GB),GAZIBORA KHAN (GB),10/04/2025,İzmir Şirinyer Hipodromu,2,sand,Good Going,1,57.0,ALPEREN OLUK,1900,129.89,6,55.30,56.61


In [32]:
#Create new table with selected columns
selected_columns = [
    "name",
    "group",
    "surface",
    "ground",
    "combined_breed_score",
    "jockey_score",
    "weight",
    "metre",
    "time"
]

df = df[selected_columns]
print(df.size)
df.head()

2643633


,name,group,surface,ground,combined_breed_score,jockey_score,weight,metre,time
0,Super Plane,3 Years Old Thoroughbreds,sand,Good Going,53.72,55.15,62.0,1900,127.52
1,Crazygold,3 Years Old Thoroughbreds,sand,Good Going,55.53,54.78,60.0,1900,126.83
2,Always A Reason,3 Years Old Thoroughbreds,sand,Good Going,52.97,56.51,58.0,1900,128.98
3,Aydos Kız,3 Years Old Thoroughbreds,sand,Good Going,56.18,55.11,58.0,1900,127.83
4,Başpınar,3 Years Old Thoroughbreds,sand,Good Going,55.30,56.61,57.0,1900,129.89


In [33]:
random_rows = df.sample(n=50)
random_rows.head()

#count = df['surface'].nunique()
#print(count)

,name,group,surface,ground,combined_breed_score,jockey_score,weight,metre,time
153940,Arya Kuş,3 Years Old Thoroughbreds,grass,Good Going,58.67,56.26,55.0,1900,122.45
291172,Royaliz,2 Years Old Thoroughbreds,grass,Good Going,52.83,53.98,52.1,1400,85.99
124124,Winter Sunshine,3 Years Old Thoroughbreds,sand,Good Going,53.92,54.12,60.5,2000,125.38
137394,Ceydacanım,3 Years Old Purebred Arabians,sand,Good Going,57.55,59.25,58.0,1200,91.99
22703,Big Gamble,3 Years Old And Up Thoroughbreds,grass,Good Going,51.95,54.19,58.0,1300,79.64


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.losses import MeanSquaredError
from sklearn.metrics import mean_absolute_error

# Fix numerics
numeric_cols = ["combined_breed_score", "jockey_score", 'weight', 'metre', 'time']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', '').str.strip(), errors='coerce')
df = df.dropna(subset=numeric_cols)

# Categorical columns
cat_cols = ['name', 'group', 'surface', 'ground']
num_cols = ['combined_breed_score', 'jockey_score', 'weight', 'metre']

# Features and label
X = df[cat_cols + num_cols]
y = df["time"]

#possibilities

# Preprocessing: OneHot for categoricals, Scale numerics
preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown='ignore'), cat_cols),
    ("num", StandardScaler(), num_cols)
])

X_processed = preprocessor.fit_transform(X)

# Initialize KFold cross-validation
kf = KFold(n_splits=3, shuffle=True, random_state=42)

# Initialize list to store MAE for each fold
mae_scores = []

model = None

# K-fold cross-validation
for train_index, val_index in kf.split(X_processed):
    X_train, X_val = X_processed[train_index], X_processed[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]

    # Build Neural Network model
    if model is None:
        model = Sequential([
            Input(shape=(X_train.shape[1],)),
            Dense(256, activation='relu'),
            BatchNormalization(),
            Dropout(0.2),
            Dense(128, activation='sigmoid'),
            BatchNormalization(),
            Dropout(0.1),
            BatchNormalization(),
            Dense(64, activation='tanh'),
            BatchNormalization(),
            Dense(1)
        ])

        model.compile(
            optimizer=Adam(learning_rate=0.001),
            loss=MeanSquaredError(),
            metrics=['mae']
        )

    #early_stop = EarlyStopping(patience=10, restore_best_weights=True)

    # Train the model
    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=3,
        batch_size=16,
        # callbacks=[early_stop],
        verbose=1
    )

    # Evaluate the model on validation set and store MAE
    y_val_pred = model.predict(X_val)
    mae = mean_absolute_error(y_val, y_val_pred)
    mae_scores.append(mae)
    print(f"Fold MAE: {mae:.2f} seconds")

# Calculate the average MAE across all folds
avg_mae = np.mean(mae_scores)
print(f"\nAverage MAE across all folds: {avg_mae:.2f} seconds")


Epoch 1/3
 4851/12239 ━━━━━━━━━━━━━━━━━━━━ 11:30 94ms/step - loss: 3790.4836 - mae: 44.9123

In [ ]:
# Save the model (full model: architecture + weights + optimizer)
model.save("race_time_model-v3.keras")
print("✅ Model saved as race_time_model.keras")

In [ ]:
import joblib

# Save the preprocessor pipeline
joblib.dump(preprocessor, "preprocessor-v3.pkl")
print("✅ Preprocessor saved as preprocessor.pkl")

In [ ]:
# Create a new input DataFrame
new_data = pd.DataFrame([{
    "name": "Naldökenli",
    "group": "4 Years Old And Up Purebred Arabians",
    "surface": "sand",
    "ground": "Good Going",
    "combined_breed_score": 59.62,
    "jockey_score": 55.27,
    "weight": 57.0,
    "metre": 1400
}])

# Preprocess and predict
new_processed = preprocessor.transform(new_data)
predicted_time = model.predict(new_processed)
print("Predicted Time:", round(predicted_time[0][0], 2), "seconds")


In [ ]:
from tensorflow.keras.models import load_model
import joblib
import pandas as pd

# Load the model and preprocessor
model_new = load_model("race_time_model-v3.keras")
preprocessor_new = joblib.load("preprocessor-v3.pkl")

# Example: New input
new_data = pd.DataFrame([{
    "name": "Beautiful Mina",
    "group": "3 Years Old Thoroughbreds",
    "mother": "Mİ BOMBON",
    "father": "SUPER SAVER (USA)",
    "surface": "sand",
    "ground": "Good Going",
    "weight": 57.0,
    "jockey": "AHMET ÇELİK",
    "metre": 1400
}])

# Transform and predict
new_processed = preprocessor_new.transform(new_data)
predicted_time = model_new.predict(new_processed)

print("🏁 Predicted Time:", round(predicted_time[0][0], 2), "seconds")

In [ ]:
#Calculate the nonlineer effect of metre on time
import matplotlib.pyplot as plt
import seaborn as sns

sns.scatterplot(x=df["metre"], y=df["time"])
plt.title("Metre vs Time")
plt.xlabel("Distance (metre)")
plt.ylabel("Race Time (seconds)")
plt.grid(True)
plt.show()

In [ ]:
#Plot predicted vs actual times:
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.2, random_state=42)

y_pred = model.predict(X_test)

plt.scatter(y_test, y_pred)
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Prediction Accuracy")
plt.grid(True)

In [ ]:
query = "SELECT * FROM `tahminler` WHERE `racedate` = 20250410"
try:
    mydb = connection.connect(host="taner.web.tr", database = 'tanerweb_tjk',user="tanerweb_tjk", passwd="Ka9jVjRJRtRW",use_pure=True)
    df = pd.read_sql(query,mydb)
    mydb.close() #close the connection
except Exception as e:
    mydb.close()
    print(str(e))

df["predicted_time"] = ""

# Fix numerics
numeric_cols = ['weight', 'metre', 'time']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', '').str.strip(), errors='coerce')
df = df.dropna(subset=numeric_cols)

# Categorical columns
cat_cols = ['name', 'group', 'mother', 'father', 'surface', 'ground', 'jockey']
num_cols = ['weight', 'metre']

# Features and label
pX = df[cat_cols + num_cols]

df.head()

In [ ]:
# Load the model and preprocessor
model_new = load_model("race_time_model-v2.keras")
preprocessor_new = joblib.load("preprocessor-v2.pkl")

# Transform and predict
for index, row in pX.iterrows():
    new_data = pd.DataFrame([row])
    new_processed = preprocessor_new.transform(new_data)
    predicted_time = model_new.predict(new_processed)
    df.at[index, "predicted_time"] = round(predicted_time[0][0], 2)

df.head()

In [ ]:
df